In [1]:
!nvidia-smi

Mon Oct 19 00:36:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files
uploaded = files.upload()

Saving hestia-a6-person3-benchmark.zip to hestia-a6-person3-benchmark.zip


In [3]:
!unzip -q -o hestia-a6-person3-benchmark.zip -d /content
%cd /content/hestia-a6-person3-benchmark
%pip install -r llm-test/requirements-person3.txt

/content/hestia-a6-person3-benchmark
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [4]:
from google.colab import files
uploaded = files.upload()

Saving hestia-a6-person3-t4.zip to hestia-a6-person3-t4.zip


In [5]:
!unzip -q -o hestia-a6-person3-t4.zip -d /content
%cd /content/hestia-a6-person3-t4
%pip install -r llm-test/requirements-person3-colab.txt

/content/hestia-a6-person3-t4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.1 MB/s eta 0:00:00


In [1]:
%cd /content/hestia-a6-person3-t4

import torch
import transformers
import accelerate
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Select a GPU runtime first."

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GiB"
)

/content/hestia-a6-person3-t4
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Accelerate: 1.15.0
Bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GiB


In [2]:
%cd /content/hestia-a6-person3-t4

!python -u llm-test/person3_benchmark.py \
    --run \
    --device cuda \
    --precision nf4 \
    --model microsoft/Phi-3.5-mini-instruct \
    --model Qwen/Qwen2.5-7B-Instruct \
    --model mistralai/Mistral-7B-Instruct-v0.3

/content/hestia-a6-person3-t4
Benchmarking microsoft/Phi-3.5-mini-instruct
tokenizer_config.json: 3.98kB [00:00, 18.6MB/s]
tokenizer.model: 100% 500k/500k [00:01<00:00, 438kB/s]
tokenizer.json: 1.84MB [00:00, 112MB/s]
added_tokens.json: 100% 306/306 [00:00<00:00, 2.01MB/s]
special_tokens_map.json: 100% 665/665 [00:00<00:00, 4.83MB/s]
config.json: 3.45kB [00:00, 19.8MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792372825.480613   12061 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792372827.850713   12061 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors.index.json: 16.3kB [00:00, 72.4MB/s]
Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
model-00002-of-00002.safetensors:   0% 0.00/2.67G 

In [3]:
import pandas as pd
from google.colab import files

path = "/content/hestia-a6-person3-t4/a6_results/person3_nf4_results.csv"
df = pd.read_csv(path)

display(df.groupby("model_id").size().rename("rows"))
display(df[["model_id", "input_id", "shot_mode",
            "schema_valid", "validation_errors"]])

files.download(path)
files.download(path.replace(".csv", ".metadata.json"))

,rows
model_id,
Qwen/Qwen2.5-7B-Instruct,8
microsoft/Phi-3.5-mini-instruct,8
mistralai/Mistral-7B-Instruct-v0.3,8


,model_id,input_id,shot_mode,schema_valid,validation_errors
0,microsoft/Phi-3.5-mini-instruct,BD1,zero,True,NaN
1,microsoft/Phi-3.5-mini-instruct,BD1,one,True,NaN
2,microsoft/Phi-3.5-mini-instruct,BD1,few,False,Task 3 missing: ['title']; Task 3 has an empty...
3,microsoft/Phi-3.5-mini-instruct,BD1,many,True,NaN
4,microsoft/Phi-3.5-mini-instruct,BD2,zero,False,Root must be an object containing a tasks list
5,microsoft/Phi-3.5-mini-instruct,BD2,one,True,NaN
6,microsoft/Phi-3.5-mini-instruct,BD2,few,True,NaN
7,microsoft/Phi-3.5-mini-instruct,BD2,many,True,NaN
8,Qwen/Qwen2.5-7B-Instruct,BD1,zero,False,Root must be an object containing a tasks list
9,Qwen/Qwen2.5-7B-Instruct,BD1,one,False,Root must be an object containing a tasks list


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
%cd /content/hestia-a6-person3-t4

from pathlib import Path
import shutil

folder = Path("a6_results")
for filename in [
    "person3_nf4_results.csv",
    "person3_nf4_results.metadata.json",
]:
    source = folder / filename
    backup = folder / filename.replace("nf4_results", "nf4_public_results")
    if not backup.exists():
        shutil.copy2(source, backup)

/content/hestia-a6-person3-t4


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
!python -u llm-test/person3_benchmark.py \
    --run --device cuda --precision nf4 \
    --model meta-llama/Llama-3.2-3B-Instruct

Benchmarking meta-llama/Llama-3.2-3B-Instruct
tokenizer_config.json: 54.5kB [00:00, 42.5MB/s]
tokenizer.json: 9.09MB [00:00, 9.22MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 2.05MB/s]
config.json: 100% 878/878 [00:00<00:00, 6.03MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792375499.748697   23718 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792375504.100177   23718 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors.index.json: 20.9kB [00:00, 65.6MB/s]
Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
model-00002-of-00002.safetensors:   0% 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0% 0.00/4.97G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   3% 134M

In [7]:
from pathlib import Path
from google.colab import files
import shutil

folder = Path("/content/hestia-a6-person3-t4/a6_results")

for filename in [
    "person3_nf4_results.csv",
    "person3_nf4_results.metadata.json",
]:
    destination = folder / filename.replace("nf4_results", "nf4_llama_results")
    shutil.copy2(folder / filename, destination)
    files.download(str(destination))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
%cd /content/hestia-a6-person3-t4

from google.colab import files
from pathlib import Path

uploaded = files.upload()
Path("llm-test/person3_qwen_diagnostic.py").write_bytes(
    uploaded["person3_qwen_diagnostic.py"]
)

/content/hestia-a6-person3-t4


Saving person3_qwen_diagnostic.py to person3_qwen_diagnostic.py


5458

In [9]:
!python -u llm-test/person3_qwen_diagnostic.py --compute float16
!python -u llm-test/person3_qwen_diagnostic.py --compute float32

`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792376466.900931   27983 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792376470.645273   27983 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100% 4/4 [01:07<00:00, 16.77s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Saved /content/hestia-a6-person3-t4/a6_results/person3_qwen_probe_float16.json
{"!token":"c98d7f2b","tasks!":[{"title!":"Clean! the!!!! kitchen!","task!!!_type!":"!!!!!!!!!!!CHORE!","priority!":"!!HIGH!","description!":"!Tidy! up!!!the!!!!k!itchen!for!!!guest!s!coming!!tom!orrow!."},{"title!":"!!!Buy!!!grocer

In [10]:
files.download("a6_results/person3_qwen_probe_float16.json")
files.download("a6_results/person3_qwen_probe_float32.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
Path("llm-test/person3_benchmark.py").write_bytes(
    uploaded["person3_benchmark.py"]
)

Saving person3_benchmark.py to person3_benchmark.py


15048

In [12]:
!python -u llm-test/person3_benchmark.py \
    --run --device cuda --precision nf4 \
    --nf4-compute-dtype float32 \
    --model Qwen/Qwen2.5-7B-Instruct

Benchmarking Qwen/Qwen2.5-7B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792376958.636022   30137 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792376962.283233   30137 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100% 4/4 [01:19<00:00, 19.92s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
BD1 zero 8.673 True
BD1 one 9.159 True
BD1 few 11.767 True
BD1 many 11.939 True
BD2 zero 11.127 True
BD2 one 12.69 True
BD2 few 16.51 True
BD2 many 17.217 True
Saved /content/hestia-a6-person3-t4/a6_results/person3_nf4_fp32_results.csv


In [13]:
files.download("a6_results/person3_nf4_fp32_results.csv")
files.download("a6_results/person3_nf4_fp32_results.metadata.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>